# Test notebook for DPD funcionality

In [ ]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
from flowermd.library import PPS

molecules = PPS(num_mols=100, lengths=50)
molecules.coarse_grain(beads={"_A": "c1cc(S)ccc1"})
molecules.bond_length

# perfomance issue in this building/CG step for large systems

## New Random Walk System Class

In [ ]:
from flowermd.library import RandomWalk
import unyt as u

ref_length = 0.3438 * u.Unit("nm")
ref_mass = 32.06 * u.Unit("amu")
ref_energy = 1.065 * u.Unit("kJ/mol")
ref_values_dict = {"length": ref_length, "mass": ref_mass, "energy": ref_energy}

system = RandomWalk(
    molecules=molecules,
    density=1.32 * u.Unit("g/cm**3"),
    bond_length=1.4226,
    buffer=0.58,
    base_units=ref_values_dict,
)

## FF from GMSO XML file

In [ ]:
from flowermd.library.forcefields import Bead_Spring_DPD

In [ ]:
system.apply_forcefield(force_field=Bead_Spring_DPD(), r_cut=1.15, kT=1.0)

## Forcefield class

from flowermd.library import DPD

A = 2000
gamma = 1500
kT = 1.0
r_cut = 1.4226
bond_k = 50000
bond_r0 = 1.4226
dpd_ff = DPD(
    A=A, gamma=gamma, kT=kT, r_cut=r_cut, bond_k=bond_k, bond_r0=bond_r0
)

In [ ]:
from flowermd.library import PhantomWalk

A = 2000
gamma = 1500
kT = 1.0
r_cut = 1.4226
bond_k = 50000
bond_r0 = 1.4226
sim = PhantomWalk(
    initial_state=system.hoomd_snapshot,
    forcefield=system.hoomd_forcefield,
    gsd_write_freq=10,
    log_write_freq=50,
    n_steps_dpd=500,
    n_steps_fire=100,
)

In [ ]:
import hoomd

for writer in sim.operations.writers:
    if isinstance(writer, hoomd.write.GSD):
        writer.flush()

In [ ]:
# system.to_gsd("random_walk_test.gsd")